# Import 

In [1]:
# Standard library imports
import os
import sys
from datetime import date

# Third-party imports
import pandas as pd
import unicodedata
import re

# Local application imports
sys.path.append('..')
sys.path.append('../..')

from file_management import get_files_dir, check_save_file
from normalize_product import *

# Get file directories
_, INPUT_DIR, OUTPUT_DIR = get_files_dir()

# Input and output files

## Input

In [2]:
# Articles with predicted products
file = OUTPUT_DIR+'/Articles/filtered_metabolic_eng_articles_with_products_cleaned_V_2025_09_30.json'
org_prod = pd.read_json(file)

org_prod = org_prod.rename(columns={'Acid_antibiotic_normalized_cleaned':'Product'}).dropna(subset=['Product'])

org_prod.head()

,Title,Abstract,Journal,Year,PMC_ID,DOI,Type,Author,Text,Product_Source,Doc_text,Product
10618204,Increased production of zeaxanthin and other p...,The psbAII locus was used as an integration pl...,Applied and environmental microbiology,2000,91786.0,10.1128/AEM.66.1.64-72.2000,"Journal Article,Research Support, U.S. Gov't, ...","[ForeName:D,LastName:Lagarde] [ForeName:L,Last...",None,title,production of zeaxanthin and,[zeaxanthin]
10618209,Expression of Alcaligenes eutrophus flavohemop...,Expression of the vhb gene encoding hemoglobin...,Applied and environmental microbiology,2000,91791.0,10.1128/AEM.66.1.98-104.2000,"Journal Article,Research Support, Non-U.S. Gov't,","[ForeName:A D,LastName:Frey] [ForeName:J E,Las...",None,abstract,heme proteins production,[heme proteins]
10649237,Altered regulation of pyruvate kinase or co-ov...,Glycolytic fluxes in resting Escherichia coli ...,Biotechnology and bioengineering,2000,NaN,10.1002/(sici)1097-0290(20000305)67:5<623::aid...,"Journal Article,Research Support, Non-U.S. Gov't,","[ForeName:M,LastName:Emmerling] [ForeName:J E,...",None,abstract,ethanol production rates,[ethanol]
10649449,Cloning and characterization of the Yarrowia l...,The squalene synthase (SQS) gene encodes a key...,"Yeast (Chichester, England)",2000,NaN,10.1002/(SICI)1097-0061(200002)16:3<197::AID-Y...,"Journal Article,Research Support, Non-U.S. Gov...","[ForeName:S,LastName:Merkulov] [ForeName:F,Las...",None,abstract,sterol biosynthesis,[sterol]
10653745,A novel genetically engineered pathway for syn...,A new pathway to synthesize poly(hydroxyalkano...,Applied and environmental microbiology,2000,91890.0,10.1128/AEM.66.2.739-743.2000,"Journal Article,Research Support, Non-U.S. Gov't,","[ForeName:S J,LastName:Liu] [ForeName:A,LastNa...",None,title,A genetically route for production of poly(hyd...,[poly(hydroxyalkanoic-acids)]


In [3]:
# ChEbi database to normalize
chebi_db = pd.read_json('../../01_Create_init_files/Products/rm_chebi.json')
chebi_db.head()

,NAME,Synonym
3,((R)-3-Hydroxybutanoyl)(n-2),[((R)-3-Hydroxybutanoyl)(n-2)]
7,(+)-car-3-ene,"[(+)-3-Carene, (1S,6R)-3,7,7-trimethylbicyclo[..."
8,(+)-8-hydroxycalamenene,"[(+)-8-Hydroxycalamenene, (5R,8S)-3,8-dimethyl..."
9,(+)-Adlumine,[(+)-Adlumine]
10,(+)-Atherospermoline,[(+)-Atherospermoline]


## Output

In [4]:
general_name = 'filtered_metabolic_eng_articles_with_products_cleaned_norm'

today = date.today()
today = today.strftime("%Y_%m_%d")

output_file = f'{general_name}_V_{today}.json'
output_file

'filtered_metabolic_eng_articles_with_products_cleaned_norm_V_2025_09_30.json'

In [5]:
org_prod = org_prod.rename(columns={'Acid_antibiotic_normalized_cleaned':'Product'}).dropna(subset=['Product'])

In [6]:
# First normalize if the name is there but also the abbreviation

# Prepare data from chebi

lets remove (+) and (-) and -, also , 

In [7]:
Treated_info = chebi_db.copy()
Treated_info.head()

,NAME,Synonym
3,((R)-3-Hydroxybutanoyl)(n-2),[((R)-3-Hydroxybutanoyl)(n-2)]
7,(+)-car-3-ene,"[(+)-3-Carene, (1S,6R)-3,7,7-trimethylbicyclo[..."
8,(+)-8-hydroxycalamenene,"[(+)-8-Hydroxycalamenene, (5R,8S)-3,8-dimethyl..."
9,(+)-Adlumine,[(+)-Adlumine]
10,(+)-Atherospermoline,[(+)-Atherospermoline]


In [8]:
# Normalize names and keep the original name
Treated_info.loc[:,'Original_name'] = Treated_info.loc[:,'NAME']
Treated_info.head()

,NAME,Synonym,Original_name
3,((R)-3-Hydroxybutanoyl)(n-2),[((R)-3-Hydroxybutanoyl)(n-2)],((R)-3-Hydroxybutanoyl)(n-2)
7,(+)-car-3-ene,"[(+)-3-Carene, (1S,6R)-3,7,7-trimethylbicyclo[...",(+)-car-3-ene
8,(+)-8-hydroxycalamenene,"[(+)-8-Hydroxycalamenene, (5R,8S)-3,8-dimethyl...",(+)-8-hydroxycalamenene
9,(+)-Adlumine,[(+)-Adlumine],(+)-Adlumine
10,(+)-Atherospermoline,[(+)-Atherospermoline],(+)-Atherospermoline


In [9]:
Treated_info.NAME = format_series_products(Treated_info.NAME)
Treated_info.NAME = format_series_products(Treated_info.NAME)

In [10]:
Treated_info = Treated_info.explode(column='Synonym')
Treated_info.head()

,NAME,Synonym,Original_name
3,3hydroxybutanoyln2,((R)-3-Hydroxybutanoyl)(n-2),((R)-3-Hydroxybutanoyl)(n-2)
7,car3ene,(+)-3-Carene,(+)-car-3-ene
7,car3ene,"(1S,6R)-3,7,7-trimethylbicyclo[4.1.0]hept-3-ene",(+)-car-3-ene
7,car3ene,"(1S)-3,7,7-trimethylbicyclo[4.1.0]hept-3-ene",(+)-car-3-ene
7,car3ene,(1S)-(+)-3-carene,(+)-car-3-ene


In [11]:
chebi_synonyms = Treated_info.set_index('Original_name')

# Prepare data from products

In [12]:
org_prod.head().loc[:,['Title','Product']]

,Title,Product
10618204,Increased production of zeaxanthin and other p...,[zeaxanthin]
10618209,Expression of Alcaligenes eutrophus flavohemop...,[heme proteins]
10649237,Altered regulation of pyruvate kinase or co-ov...,[ethanol]
10649449,Cloning and characterization of the Yarrowia l...,[sterol]
10653745,A novel genetically engineered pathway for syn...,[poly(hydroxyalkanoic-acids)]


In [13]:
remove_and = r'\sand$'
test_products = org_prod.Product.explode().str.rstrip().str.replace(remove_and, '', regex = True).str.lower()

test_products.head()

10618204                     zeaxanthin
10618209                  heme proteins
10649237                        ethanol
10649449                         sterol
10653745    poly(hydroxyalkanoic-acids)
Name: Product, dtype: object

In [14]:
test_products.loc[test_products.str.len()<1]

Series([], Name: Product, dtype: object)

In [15]:

# remove cellulosate ethanol, just let it be ethanol
test_products = test_products.str.replace(r'\b(ligno)?cellulosic\b','',regex=True)
test_products = test_products.str.replace(r'\bpure\b','',regex=True)
test_products = test_products.str.replace(r'\bhuman\b','',regex=True)

test_products = test_products.str.replace(r'\b(bio)(fuel|ethanol|butanol|diesel|ethylene|lipid)s?\b',r'\2',regex=True)
#test_products = test_products.str.replace(r'\b(bio)?(polymer)\b',r'',regex=True)



In [16]:
test_products.loc[test_products.str.len()<1]

Series([], Name: Product, dtype: object)

In [17]:
#test_products = test_products.str.replace(regular_expression, '', regex = True).str.lower()
test_products = format_series_products(test_products)
test_products = format_series_products(test_products)
test_products = test_products.str.strip(GENERAL_STRIP)

In [18]:
test_products.loc[test_products.str.len()<1]
test_products = test_products.replace('', pd.NA)  # For pandas 1.0+
test_products = test_products.dropna()

In [19]:
test_products = test_products.apply(replace_greek_letters_with_regex)

In [20]:
import logging
from collections import defaultdict

# Configure logging once (e.g. at top of your script)
logging.basicConfig(
    level=logging.INFO,  # you can switch to DEBUG for more detail
    format="%(message)s"
)

# Dictionary to track unmatched values
unmatched_counts = defaultdict(int)

def get_value(value, chebi_df):
    new_value = chebi_df.index[chebi_df['Synonym'] == value]
    if not new_value.empty:
        return new_value[0]
    
    elif value.strip(GENERAL_STRIP).endswith('s') and value != 'biomass':
        new_value = chebi_df.index[chebi_df['Synonym'] == value[:-1]]
        if not new_value.empty:
            return new_value[0]
        else:
            logging.debug(f"No match (plural stripped): {value}")
            unmatched_counts[value] += 1
            return value
    else:
        logging.info(f"No match found: {value}")
        unmatched_counts[value] += 1
        return value

# Function to get the most common unmatched values
def get_most_common_unmatched(n=10):
    """Return the n most common unmatched values and their counts"""
    sorted_unmatched = sorted(unmatched_counts.items(), key=lambda x: x[1], reverse=True)
    return sorted_unmatched[:n]

# Function to print unmatched statistics
def print_unmatched_stats():
    """Print statistics about unmatched values"""
    total_unmatched = sum(unmatched_counts.values())
    unique_unmatched = len(unmatched_counts)
    
    print(f"Total unmatched values: {total_unmatched}")
    print(f"Unique unmatched values: {unique_unmatched}")
    
    if unique_unmatched > 0:
        print("\nTop 10 most common unmatched values:")
        for value, count in get_most_common_unmatched(10):
            print(f"  {value}: {count} times")

In [21]:
from clean_products import *


In [22]:
joined_data_rejoined = test_products.groupby(level=0).agg(list)

In [23]:
joined_data_rejoined = pd.DataFrame(joined_data_rejoined)

In [24]:
joined_data_rejoined

,Product
10618204,[zeaxanthin]
10618209,[hemeproteins]
10649237,[ethanol]
10649449,[sterol]
10653745,[polyhydroxyalkanoicacids]
...,...
40572067,[amylosucrase]
40572208,[nitrogenase]
40573728,[cucurbitanetypemogrosides]
40577193,[ethanol]


In [25]:
removed_log=[]
joined_data_rejoined['Product'] = joined_data_rejoined.apply(
    lambda row: clean_product_list(row.name, row['Product'],removed_log), axis=1
)

In [26]:
import logging

# Set up logging (you can configure this at the top of your notebook)
logging.basicConfig(level=logging.INFO, format='%(message)s')
logger = logging.getLogger(__name__)

if not removed_log:
    logger.info("✅ No fake products (dates, numbers, vol.#, suffix words) found.")
else:
    logger.info(f"❌ Found and removed fake products from {len(removed_log)} rows:\n")
    for entry in removed_log:
        idx = entry['row_index']
        original = entry['original_list']
        removed = entry['removed_items']
        logger.info(f"Row {idx} — original list: {original}")
        for val, reason in removed:
            logger.info(f"   ⟶ Removed: '{val}'  ({reason})")
        logger.info("")  # Empty line for separation

❌ Found and removed fake products from 40 rows:

Row 10931852 — original list: ['fattyacid', '1128']
   ⟶ Removed: '1128'  (plain-number)

Row 15575692 — original list: ['purification']
   ⟶ Removed: 'purification'  (removed leading suffix word)
   ⟶ Removed: 'purification'  (all words removed by suffix rules)

Row 17275940 — original list: ['iv', 'seibert', '2006']
   ⟶ Removed: '2006'  (plain-number)

Row 19374996 — original list: ['20']
   ⟶ Removed: '20'  (plain-number)

Row 21705610 — original list: ['5478']
   ⟶ Removed: '5478'  (plain-number)

Row 22443545 — original list: ['1962']
   ⟶ Removed: '1962'  (plain-number)

Row 22639141 — original list: ['3']
   ⟶ Removed: '3'  (plain-number)

Row 22886996 — original list: ['6']
   ⟶ Removed: '6'  (plain-number)

Row 22952739 — original list: ['excessmethylglyoxal', '25']
   ⟶ Removed: '25'  (plain-number)

Row 23500000 — original list: ['3']
   ⟶ Removed: '3'  (plain-number)

Row 23866108 — original list: ['10']
   ⟶ Removed: '10'  

In [27]:
test_products.loc[test_products.str.len()<1]

Series([], Name: Product, dtype: object)

In [28]:
joined_data_rejoined['Product'] = joined_data_rejoined['Product'].apply(split_acid_related_products)

In [29]:
joined_data_rejoined['Product'] = joined_data_rejoined['Product'].apply(
    lambda x: np.nan if isinstance(x, list) and len(x) == 1 and pd.isna(x[0]) else x
)

In [30]:
joined_data = joined_data_rejoined

In [31]:
import numpy as np

In [32]:
def acid_ate(lista):
    new_lista = []
    for text in lista:
        new_text = re.sub(r'ic acid\b', 'ate', text)
        new_text = re.sub(r'ate acid\b', 'ate', new_text)

        new_text = re.sub(r'\bamino\b\s?$', 'amino acid', new_text)
        #new_text = re.sub(r'\bpolycyclic tetramate\b\s?$', 'polycyclic tetramate macrolactams', new_text)

        new_lista.append(new_text)
    return new_lista

In [33]:
# Change 'ic acid' to 'ate' and amino alone to amino acid
joined_data['Acid_normalized'] = joined_data['Product'].apply(
    lambda x: acid_ate(x) if isinstance(x, list) else np.nan
)

In [34]:
joined_data['Acid_normalized'] = joined_data['Acid_normalized'].apply(split_acid_related_products)

In [ ]:
joined_data['Acid_antibiotic_normalized'] = joined_data['Acid_normalized'].apply(split_antibiotic_related_products)

In [36]:
top_terms = joined_data.Acid_antibiotic_normalized.explode().dropna().value_counts().head(150).index
top_terms = set(top_terms)
top_terms = top_terms- {'protein','proteins','enzyme','enzymes','aromatic','peptide','peptides',
                        'gene','growth','lipid','lipids'} #special case lipid


# Now apply to your dataframe
joined_data['Acid_antibiotic_normalized_cleaned'] = joined_data['Acid_antibiotic_normalized'].apply(
    lambda x: split_if_multiple_top_terms(x, top_terms)[0] if isinstance(x, list) else x
)

In [37]:
# ── 3. Apply the cleaning to the DataFrame ───────────────────────────────
removed_log=[]
joined_data['Acid_antibiotic_normalized_cleaned'] = joined_data.apply(
    lambda row: clean_product_list(row.name, row['Acid_antibiotic_normalized_cleaned'],removed_log),
    axis=1
)

# ── 4. Report the removals with full context ─────────────────────────────

if not removed_log:
    logger.info("✅ No fake products (dates, numbers, vol.#, suffix words) found.")
else:
    logger.info(f"❌ Found and removed fake products from {len(removed_log)} rows:\n")
    for entry in removed_log:
        idx = entry['row_index']
        original = entry['original_list']
        removed = entry['removed_items']
        logger.info(f"Row {idx} — original list: {original}")
        for val, reason in removed:
            logger.info(f"   ⟶ Removed: '{val}'  ({reason})")
        logger.info("")  # Empty line for separation

✅ No fake products (dates, numbers, vol.#, suffix words) found.


In [38]:
import pandas as pd
import numpy as np

# your fake values list (already deduplicated)
fake_values = [
    "towards","difficult","mixture","role","method","standpoint","certain",
    "several","future","overall","degree","suite","procedure","repertoire",
    "recovery","maincatalyst","strategy","discovery","performance",
    "superiorperformance","mechanism","response","interplay","spectrum",
    "candidate","technique","could","proxy","entire","academic","widespread",
    "healthy","costly","despite","dual","myriad","mainobstacle","bilevel",
    "heterotroph","organic","toolbox","wildtype","choice","data","toward",
    "length","survival","criterion","biology","immensevariety","latter",
    "shortcut","frontier","scaleup","bulkscale","grand","common","universal",
    "global","addedvalue","recalcitrance","storage","palette","mode",
    "productivity","machinery","attenuation","redox","phenotypic","fossil",
    "variety","vast","sensor","authentic","revision","scenario","dropin",
    "since","delivery","barrier","portfolio","disease","workhorse","result",
    "phenotype","biotic","mimicry","efficacy","unknown","covert",
    "gatekeeper","native","area","anthropogenic","formula","basic",
    "enantiomeric","problem","cheap","may","chapter","acidic","superior",
    "wealth","scope","benign","ton","juice","proven","meaningful","concise",
    "performer","rand","kind","division","resource","wastewater", "main",
] + ['lignocellulosic','cellulosic','pure','human']

# ensure it's a set for faster lookup
fake_set = set(fake_values)

# function to clean each row (with printing if fake found)
def clean_list_with_logging(values):
    if not isinstance(values, list):  # skip NaN or non-list
        return values
    
    cleaned = [v for v in values if v not in fake_set]

    # check if anything was removed
    if len(cleaned) != len(values):
        print("⚠️ Found fake value(s)!")
        print("Original list:", values)

        print("-" * 60)

    return cleaned if cleaned else np.nan

# apply cleaning with both columns
joined_data["Acid_antibiotic_normalized_cleaned"] = joined_data.apply(
    lambda row: clean_list_with_logging(row["Acid_antibiotic_normalized_cleaned"]),
    axis=1
)

⚠️ Found fake value(s)!
Original list: ['bilevel']
------------------------------------------------------------
⚠️ Found fake value(s)!
Original list: ['wildtype']
------------------------------------------------------------
⚠️ Found fake value(s)!
Original list: ['bulkscale']
------------------------------------------------------------
⚠️ Found fake value(s)!
Original list: ['scaleup']
------------------------------------------------------------
⚠️ Found fake value(s)!
Original list: ['addedvalue']
------------------------------------------------------------
⚠️ Found fake value(s)!
Original list: ['methylpropionate', 'wildtype']
------------------------------------------------------------
⚠️ Found fake value(s)!
Original list: ['dropin']
------------------------------------------------------------
⚠️ Found fake value(s)!
Original list: ['addedvalue']
------------------------------------------------------------
⚠️ Found fake value(s)!
Original list: ['dropin', 'fuel']
-----------------

In [39]:
test_products = joined_data.Acid_antibiotic_normalized_cleaned.explode().dropna()

In [40]:
chebi_synonyms[chebi_synonyms.index.str.contains('microbial')]

,NAME,Synonym
Original_name,,
antimicrobial agent,antimicrobialagent,antibiotics
antimicrobial agent,antimicrobialagent,antibiotic
antimicrobial agent,antimicrobialagent,antimicrobial
antimicrobial agent,antimicrobialagent,antimicrobials
antimicrobial agent,antimicrobialagent,microbicides
...,...,...
EC 3.4.24.3 (microbial collagenase) inhibitor,ec34243microbialcollagenaseinhibitor,microbial collagenase inhibitor
EC 3.4.24.3 (microbial collagenase) inhibitor,ec34243microbialcollagenaseinhibitor,microbial collagenase (EC 3.4.24.3) inhibitor
Liver-expressed antimicrobial peptide 1,liverexpressedantimicrobialpeptide1,LEAP-1


In [41]:
chebi_synonyms[chebi_synonyms.index=='conate']

,NAME,Synonym
Original_name,,


In [42]:
chebi_synonyms[chebi_synonyms.NAME=='conate']

,NAME,Synonym
Original_name,,
"cis,cis-muconate",conate,"cis,cis-muconate"
"cis,cis-muconate",conate,"(2Z,4Z)-hexa-2,4-dienedioate"


In [43]:
chebi_synonyms[chebi_synonyms.NAME=='ester']

,NAME,Synonym
Original_name,,
ester,ester,Ester
ester,ester,esters


In [44]:
chebi_synonyms[chebi_synonyms.NAME=='diaminohexanoate']

,NAME,Synonym
Original_name,,
"(3S,5S)-3,5-diaminohexanoate",diaminohexanoate,"(3S,5S)-3,5-diaminocaproate"
"(3S,5S)-3,5-diaminohexanoate",diaminohexanoate,"L-erythro-3,5-diaminocaproate"
"(3S,5S)-3,5-diaminohexanoate",diaminohexanoate,"L-erythro-3,5-diaminohexanoate"
"(3S,5S)-3,5-diaminohexanoate",diaminohexanoate,"(3S,5S)-3,5-diaminohexanoate"


In [45]:
chebi_synonyms[chebi_synonyms.NAME.str.lower().str.contains('ethylene')] 

,NAME,Synonym
Original_name,,
"(S)-N-[3-(3,4-Methylenedioxyphenyl)-2-(acetylthio)methyl-1-oxoprolyl]-(S)-alanine benzyl ester",334methylenedioxyphenyl2acetylthiomethyl1oxopr...,"(S)-N-[3-(3,4-Methylenedioxyphenyl)-2-(acetylt..."
"[3-(3,4-methylenedioxyphenyl)-2-(mercaptomethyl)-1-oxoprolyl]alanine",334methylenedioxyphenyl2mercaptomethyl1oxoprol...,"(S)-N-[3-(3,4-Methylenedioxyphenyl)-2-(mercapt..."
"[3-(3,4-methylenedioxyphenyl)-2-(mercaptomethyl)-1-oxoprolyl]alanine",334methylenedioxyphenyl2mercaptomethyl1oxoprol...,"4-(1,3-benzodioxol-5-yl)-1-hydroxy-5-(sulfanyl..."
24-methylenecycloartanol,methylenecycloartanol,24-methylenecycloartanol
24-methylenecycloartanol,methylenecycloartanol,24-methylenecycloartanol
...,...,...
N'-(1h-indol-3-ylmethylene)benzohydrazide,n'1hindol3ylmethylenebenzohydrazide,N-[(E)-1H-indol-3-ylmethylideneamino]benzamide
"5,2'-Dihydroxy-7-methoxy-6,8-dimethyl-4',5'-methylenedioxyflavan",52'dihydroxy7methoxy68dimethyl4'5'methylenedio...,"2-(6-hydroxy-1,3-benzodioxol-5-yl)-7-methoxy-6..."
"2'-Hydroxy-7-methoxy-4',5'-methylenedioxyflavan",2'hydroxy7methoxy4'5'methylenedioxyflavan,"6-[(2S)-7-methoxy-3,4-dihydro-2H-chromen-2-yl]..."


In [50]:
test_products

10618204                   zeaxanthin
10618209                 hemeproteins
10649237                      ethanol
10649449                       sterol
10653745     polyhydroxyalkanoicacids
                      ...            
40572067                 amylosucrase
40572208                  nitrogenase
40573728    cucurbitanetypemogrosides
40577193                      ethanol
40579636                          pga
Name: Acid_antibiotic_normalized_cleaned, Length: 15103, dtype: object

In [ ]:
# Define your allow list
ALLOWED_UNMATCHED = ['fuel','protein', 'butanol', 'fattyacid', 'lipid', 'fattyacids','hydroxypropionate',
                     'propanediol','lycopene','astaxanthin','biomass','diesel','aminoacid',
                     'aminoacids','enzyme','enzymes','conate']

# Create a mask for values that are in your allow list
allowed_mask = test_products.isin(ALLOWED_UNMATCHED)

# Apply the function only to values NOT in the allow list
result = test_products.copy()  # Create a copy to modify

# For values not in allow list, apply the matching function
result[~allowed_mask] = test_products[~allowed_mask].apply(
    get_value, chebi_df=chebi_synonyms
)

# Values in allow list remain unchanged
result[allowed_mask] = test_products[allowed_mask]

In [54]:
things = get_most_common_unmatched(100)

In [55]:
things = pd.DataFrame(things)
things.head()

,0,1
0,aminobutyrate,49
1,fucosyllactose,49
2,cellulase,46
3,resveratrol,41
4,phb,39


In [56]:
things.iloc[:,1].sum()

np.int64(1765)

In [57]:
result

10618204                   zeaxanthin
10618209                  hemoprotein
10649237                      ethanol
10649449                       sterol
10653745     polyhydroxyalkanoicacids
                      ...            
40572067                 amylosucrase
40572208                  nitrogenase
40573728    cucurbitanetypemogrosides
40577193                      ethanol
40579636                          pga
Name: Acid_antibiotic_normalized_cleaned, Length: 15103, dtype: object

In [58]:
result = result.str.replace('ic acid','ate')
result = result.str.replace('protein polypeptide chain','protein')#.str.rstrip('s')

In [59]:
result.value_counts().head(60)

Acid_antibiotic_normalized_cleaned
ethanol                   635
protein                   398
fuel                      334
lactate                   217
succinate(2-)             213
lipid                     161
butanol                   157
fattyacid                 141
butanediol                122
biomass                   105
fattyacids                105
carotenoid                103
poly(hydroxyalkanoate)     99
isobutanol                 97
polyketide                 91
hydroxypropionate          88
L-lysine                   87
carotene                   78
propanediol                78
acetate                    75
lycopene                   75
astaxanthin                72
itaconate(2-)              71
xylitol                    71
terpenoid                  63
riboflavin                 62
malate(2-)                 62
aminoacids                 61
antimicrobial agent        59
pyruvate                   57
enzyme                     56
glycerol                   56
enzym

In [60]:
result.value_counts()

Acid_antibiotic_normalized_cleaned
ethanol                        635
protein                        398
fuel                           334
lactate                        217
succinate(2-)                  213
                              ... 
hypercellulase                   1
invitrogendscdna                 1
microorganismsfattyacidsffa      1
areas                            1
cucurbitanetypemogrosides        1
Name: count, Length: 4906, dtype: int64

In [61]:
without_num = result.dropna()

In [63]:
without_num = format_series_products(without_num)

In [64]:
without_num = standardize_product_names(without_num)

In [65]:
without_num.str.contains('pha').sum()

np.int64(336)

In [66]:
without_num = standardize_product_names(without_num)

In [67]:
#without_num = without_num.str.replace('\bbutanediolbdo\b','butanediol', regex=True)#.str.rstrip('s')
#
without_num.value_counts()[without_num.value_counts().index.str.contains('phb')]

Acid_antibiotic_normalized_cleaned
phb                        81
phbv                        4
parahydroxybenzoatephba     1
phbhhx                      1
phbh16                      1
Name: count, dtype: int64

In [69]:
without_num = format_series_products(without_num)

In [75]:

remove_start_weird_digits = r'^[sSrRdD](\d+)'

def format_series_products(product_series):
    # Normalize subindex
    product_series = product_series.str.replace(remove_end_weird, '', regex = True)
    product_series = product_series.str.replace(remove_end_weird2, '', regex = True)

    product_series = product_series.str.replace(remove_start_weird, '', regex = True)
    product_series = product_series.str.replace(remove_start_weird2, '', regex = True)
    product_series = product_series.str.replace(remove_start_weird3, '', regex = True)
    product_series = product_series.str.replace(remove_start_weird4, '', regex = True)
    product_series = product_series.str.replace(remove_start_weird5, '', regex = True)
    product_series = product_series.str.replace(remove_start_weird6, '', regex = True)
    product_series = product_series.str.replace(remove_start_weird7, '', regex = True)
    product_series = product_series.str.replace(remove_start_weird8, '', regex = True)

    product_series = product_series.str.replace(remove_middle_weird, '', regex = True)
    product_series = product_series.str.replace(remove_middle_weird2, '', regex = True)
    
    product_series = product_series.str.replace(remove_start_weird_digits, r'\1', regex=True)

    # turn alpha into α
    product_series = product_series.str.lower().replace(inverted_all_greek,regex=True)
    product_series = product_series.str.lower().str.replace(r'^r+',r'r', regex=True)

    product_series = product_series.str.replace(isomer_racemate,'', regex=True)
    product_series = product_series.str.replace(isomer_transemate,'', regex=True)
    product_series = product_series.str.replace(isomer_cisemate,'', regex=True)

    product_series = product_series.str.replace(greek_start,'', regex=True)

    product_series = product_series.str.replace(isomer,'', regex=True)
    product_series = product_series.str.replace(isomer,'', regex=True)
    

    product_series = product_series.str.lower().str.replace(regular_expression,'', regex=True)

    product_series = product_series.str.replace(regular_expression_full,'', regex=True)
    
    product_series = product_series.apply(lambda x: unicodedata.normalize('NFKC', x) if x is not None else x)

    return(product_series)




In [76]:
# Remove l- or d- (like d-lactate) or alpha-molecule
isomer = fr'^\d*[npdolʟʀr\+{all_greek_letters}]?-'
greek_start = fr'^[{all_greek_letters}]'

isomer_racemate = r'^rac-'
isomer_transemate = r'^(trans[-,])'
isomer_cisemate = r'^(cis[-,])+'


regular_expression = r'[()\[\],\.\{\} ]'
regular_expression_full = r'[()\+\-\[\],\.\{\} ]'


remove_end_weird = r'\(-\)$'
remove_end_weird2 = r'-$'

remove_start_weird = r'^\([pRrSsNno]\)[-,]'
remove_start_weird2 = r'^[pLlDdNno]?[-,]'
remove_start_weird3 = r'^\(?\d?[RrSsdD],\d?[RrSsdD]\)?[-,]'

remove_start_weird4 = r'^ʟʀ'
remove_start_weird5 = r'^,-'
remove_start_weird6 = r'^[sSrR][sSrR]-'
remove_start_weird7 = r'^-'
remove_start_weird8 = r'^\(?\d+,\d+\)?[-,]'

remove_middle_weird = r'--'
remove_middle_weird2 = r'-,-'

remove_start_weird_digits = r'^[sSrRdD](\d+)'

remove_start_weird_digits2 = r'^(?:\d+[rsdRSdD]\s*)+([bcdfghjklmnpqrtvwxyz].*)'

def format_series_products(product_series):
    # Normalize subindex
    product_series = product_series.str.replace(remove_end_weird, '', regex = True)
    product_series = product_series.str.replace(remove_end_weird2, '', regex = True)

    product_series = product_series.str.replace(remove_start_weird, '', regex = True)
    product_series = product_series.str.replace(remove_start_weird2, '', regex = True)
    product_series = product_series.str.replace(remove_start_weird3, '', regex = True)
    product_series = product_series.str.replace(remove_start_weird4, '', regex = True)
    product_series = product_series.str.replace(remove_start_weird5, '', regex = True)
    product_series = product_series.str.replace(remove_start_weird6, '', regex = True)
    product_series = product_series.str.replace(remove_start_weird7, '', regex = True)
    product_series = product_series.str.replace(remove_start_weird8, '', regex = True)

    product_series = product_series.str.replace(remove_middle_weird, '', regex = True)
    product_series = product_series.str.replace(remove_middle_weird2, '', regex = True)
    
    product_series = product_series.str.replace(remove_start_weird_digits, r'\1', regex=True)
    product_series = product_series.str.replace(remove_start_weird_digits2, r'\1', regex=True)


    # turn alpha into α
    product_series = product_series.str.lower().replace(inverted_all_greek,regex=True)
    product_series = product_series.str.lower().str.replace(r'^r+',r'r', regex=True)

    product_series = product_series.str.replace(isomer_racemate,'', regex=True)
    product_series = product_series.str.replace(isomer_transemate,'', regex=True)
    product_series = product_series.str.replace(isomer_cisemate,'', regex=True)

    product_series = product_series.str.replace(greek_start,'', regex=True)

    product_series = product_series.str.replace(isomer,'', regex=True)
    product_series = product_series.str.replace(isomer,'', regex=True)
    

    product_series = product_series.str.lower().str.replace(regular_expression,'', regex=True)

    product_series = product_series.str.replace(regular_expression_full,'', regex=True)
    
    product_series = product_series.apply(lambda x: unicodedata.normalize('NFKC', x) if x is not None else x)

    return(product_series)


In [77]:
format_series_products(pd.Series(but.index))

0                         butanediol
1                   meso23butanediol
2                       23butanediol
3              succinate14butanediol
4                    butanediol23bdo
5                         butanediol
6               pectinase3butanediol
7          ral23butanediolbdoisomers
8                   butanediolcdw1h1
9                 stereo23butanediol
10                   butanediol14bdo
11                  2r3r23butanediol
12    homodiacetylhomoss23butanediol
13                butanediol/acetoin
Name: Acid_antibiotic_normalized_cleaned, dtype: object

In [78]:
without_num.value_counts()[without_num.value_counts().index.str.contains('valerate')]

Acid_antibiotic_normalized_cleaned
hydroxyvalerate                         9
ketoisovalerate                         7
aminovalerate                           6
biopolyamideprecursors5aminovalerate    1
hydroxyvalerate4hvamonomer              1
hydroxyisovalerate                      1
carbon55aminovalerate                   1
Name: count, dtype: int64

In [79]:
without_num[without_num.str.endswith('ly')]

24930895                 dimensionally
27403844                    societally
38672502    hypercompactcas12subfamily
38773100                   ribosomally
Name: Acid_antibiotic_normalized_cleaned, dtype: object

In [80]:
without_num.name = 'Norm_product'

In [81]:
words = without_num.value_counts().index.to_list()
for i in range(len(words)):
    for j in range(i+1, len(words)):
        if len(words[i])>1 and len(words[j])>1:

            if compare_words(words[i], words[j]):
                print(f'{words[i]} and {words[j]} are similar')

ethanol and butanol are similar
ethanol and methanol are similar
ethanol and octanol are similar
ethanol and ethanolabe are similar
ethanol and decanol are similar
ethanol and ethanol2a are similar
ethanol and ethanole2g are similar
ethanol and ethanol39 are similar
ethanol and 2gethanol are similar
ethanol and cbpethanol are similar
ethanol and ethanol58 are similar
ethanol and 2ndethanol are similar
ethanol and v/vethanol are similar
ethanol and 16ethanol are similar
ethanol and ethanol34 are similar
ethanol and indanol are similar
ethanol and nonethanol are similar
protein and protease are similar
protein and proton are similar
protein and prenol are similar
protein and rnaprotein are similar
protein and ldproteins are similar
protein and 093protein are similar
fuel and eps are similar
fuel and fig are similar
fuel and diol are similar
fuel and fad are similar
fuel and fucose are similar
fuel and urea are similar
fuel and la are similar
fuel and fdca are similar
fuel and epa are sim

hydrocarbon and spirocarbon are similar
hydrocarbon and biohydrocarbon are similar
geraniol and geranate are similar
geraniol and 53geraniol are similar
glycolate and glycerate are similar
glycolate and glycoside are similar
glycolate and glycolatega are similar
glycolate and gluconate are similar
acetylcoa and acetyltag are similar
acetylcoa and lactylcoa are similar
acetylcoa and acylcoa are similar
acetylcoa and 63acetylcoa are similar
coumarate and coumarins are similar
coumarate and coumarin are similar
coumarate and coumaratepca are similar
coumarate and coumaric are similar
alkaloid and alkanols are similar
alkane and alkene are similar
alkane and allose are similar
alkane and alkaene are similar
alkane and taxane are similar
alkane and alkanols are similar
alkane and ale are similar
alkane and bioalkane are similar
alkane and alkyl are similar
alkane and alkyne are similar
alkane and ala are similar
alkane and butane are similar
alkane and alkaenes are similar
alkane and oleate

lactone and lactonetal are similar
glutarate2 and oxoglutarate are similar
arabinitol and arabinose are similar
diterpenoid and diterpene are similar
arbutin and ambrein are similar
arbutin and rutin are similar
ginsenoside and ginsenosiderh1 are similar
ginsenoside and ginsenosidef1 are similar
ginsenoside and siamenoside are similar
ginsenoside and ginsenosiderh2 are similar
laccase and lactide are similar
laccase and vaccine are similar
laccase and lactose are similar
laccase and lactase are similar
laccase and halyase are similar
laccase and lacs are similar
h2 and co are similar
h2 and nadh are similar
h2 and 3hp are similar
h2 and pa are similar
h2 and hp are similar
h2 and gsh are similar
h2 and ga are similar
h2 and dha are similar
h2 and bio are similar
h2 and fig are similar
h2 and nin are similar
h2 and fad are similar
h2 and pga are similar
h2 and rna are similar
h2 and phas are similar
h2 and abe are similar
h2 and sam are similar
h2 and sa are similar
h2 and la are simila

methylketone and methylketones are similar
hexanoate and octanoate are similar
hexanoate and retinoate are similar
hexanoate and decanoate are similar
hexanoate and nonanoate are similar
clavulanate and tcaclavulanate are similar
pa and hp are similar
pa and gsh are similar
pa and ga are similar
pa and dha are similar
pa and bio are similar
pa and fig are similar
pa and nin are similar
pa and fad are similar
pa and pga are similar
pa and egfp are similar
pa and rna are similar
pa and phas are similar
pa and abe are similar
pa and sam are similar
pa and sa are similar
pa and la are similar
pa and phbv are similar
pa and epa are similar
pa and gfp are similar
pa and ga3 are similar
pa and ca are similar
pa and ch4 are similar
pa and c20 are similar
pa and tca are similar
pa and pnp are similar
pa and iaa are similar
pa and nps are similar
pa and cis are similar
pa and hr are similar
pa and c50 are similar
pa and nadp are similar
pa and ptm are similar
pa and br are similar
pa and eet are

bioplastic and plastics are similar
bioplastic and plastic are similar
prodigiosin and prodigiosins are similar
carboxylate and carboxysome are similar
coumarins and coumarin are similar
coumarins and coumaric are similar
glycerate and glycerol/ are similar
glycerate and gluconate are similar
steroidal and norsteroid are similar
iturin and trihb are similar
iturin and trna are similar
iturin and inulin are similar
iturin and rine are similar
cellobionate and maltobionate are similar
hydroxyisoleucine and 4s4hydroxyisoleucine are similar
pullulan and pullulanase are similar
indole3acetate and indoleacetate are similar
purine and urea are similar
purine and edeine are similar
purine and prn are similar
purine and butane are similar
purine and ines are similar
purine and purifi are similar
purine and ine are similar
purine and rine are similar
purine and offine are similar
nin and fad are similar
nin and pga are similar
nin and rna are similar
nin and abe are similar
nin and sam are simil

...

dodecanediol and dodecanols are similar
vinylgroup and enoylgroup are similar
mil and ha are similar
mil and at are similar
mil and ion are similar
mil and gla are similar
mil and mnc are similar
mil and sms are similar
mil and pl are similar
mil and pdc are similar
mil and egt are similar
mil and kg are similar
mil and b4 are similar
mil and cc are similar
mil and pe are similar
mil and ja are similar
mil and mcv are similar
mil and avi are similar
mil and ava are similar
mil and cnf are similar
mil and c22 are similar
mil and ll are similar
mil and mts are similar
mil and jd are similar
mil and pqs are similar
mil and modi are similar
mil and tmp are similar
mil and pfl are similar
mil and ndo are similar
mil and inc are similar
mil and odd are similar
mil and c18 are similar
mil and glu are similar
mil and cos are similar
mil and efe are similar
mil and efb are similar
mil and hc are similar
mil and wa are similar
mil and c≥4 are similar
mil and ota are similar
mil and bdf are simil

psag9 and pac are similar
psag9 and psca32 are similar
psag9 and pas are similar
cnf and cdhs are similar
cnf and c22 are similar
cnf and ll are similar
cnf and cfas are similar
cnf and mts are similar
cnf and jd are similar
cnf and pqs are similar
cnf and tmp are similar
cnf and pfl are similar
cnf and ndo are similar
cnf and inc are similar
cnf and adca are similar
cnf and odd are similar
cnf and c23o are similar
cnf and c18 are similar
cnf and glu are similar
cnf and cos are similar
cnf and efe are similar
cnf and efb are similar
cnf and hc are similar
cnf and wa are similar
cnf and c≥4 are similar
cnf and ota are similar
cnf and bdf are similar
cnf and se are similar
cnf and pea are similar
cnf and bia are similar
cnf and cagt are similar
cnf and bchl are similar
cnf and ctp4 are similar
cnf and gcn4 are similar
cnf and pte are similar
cnf and coq9 are similar
cnf and bem are similar
cnf and 1–3 are similar
cnf and t2 are similar
cnf and pta are similar
cnf and 4a are similar
cnf a

fcdiene and fine are similar
efb and ines are similar
efb and hc are similar
efb and wa are similar
efb and c≥4 are similar
efb and ota are similar
efb and bdf are similar
efb and meso are similar
efb and se are similar
efb and pea are similar
efb and bia are similar
efb and pte are similar
efb and bem are similar
efb and 1–3 are similar
efb and t2 are similar
efb and pta are similar
efb and 4a are similar
efb and c40 are similar
efb and crt are similar
efb and iv are similar
efb and 4ee are similar
efb and vate are similar
efb and ix are similar
efb and bm3 are similar
efb and fine are similar
efb and ¢ve are similar
efb and dag are similar
efb and iie are similar
efb and gs are similar
efb and aao are similar
efb and gb are similar
efb and sige are similar
efb and dnr are similar
efb and pd are similar
efb and whey are similar
efb and mm are similar
efb and ine are similar
efb and eryg are similar
efb and tc are similar
efb and gern are similar
efb and gerb are similar
efb and iii ar

thiol and myxol are similar
thiol and thiazole are similar
p450 and 4ee are similar
p450 and pbds are similar
p450 and pd are similar
p450 and pufa are similar
p450 and ma30 are similar
p450 and pdna are similar
p450 and pac are similar
p450 and p450s are similar
p450 and c4c5 are similar
p450 and pca are similar
p450 and ptac are similar
p450 and pas are similar
p450 and p450nov are similar
p450 and p43′ are similar
4ee and ix are similar
4ee and bm3 are similar
4ee and ¢ve are similar
4ee and dag are similar
4ee and iie are similar
4ee and gs are similar
4ee and aao are similar
4ee and gb are similar
4ee and dnr are similar
4ee and pd are similar
4ee and mm are similar
4ee and ine are similar
4ee and tc are similar
4ee and iii are similar
4ee and b0 are similar
4ee and c19 are similar
4ee and cya are similar
4ee and ccr are similar
4ee and vb1 are similar
4ee and osh are similar
4ee and mk4 are similar
4ee and cdp are similar
4ee and hrj are similar
4ee and syl are similar
4ee and mv

c19 and cya are similar
c19 and ccr are similar
c19 and vb1 are similar
c19 and osh are similar
c19 and mk4 are similar
c19 and tceg1 are similar
c19 and emcp are similar
c19 and cdp are similar
c19 and hrj are similar
c19 and mcfa are similar
c19 and rtca are similar
c19 and syl are similar
c19 and mva are similar
c19 and o8p are similar
c19 and cps are similar
c19 and gvl are similar
c19 and cb are similar
c19 and 7fa are similar
c19 and id are similar
c19 and eep are similar
c19 and c9 are similar
c19 and dof are similar
c19 and sda are similar
c19 and mw are similar
c19 and ai2 are similar
c19 and e61 are similar
c19 and bmc are similar
c19 and lacs are similar
c19 and hpa are similar
c19 and cmge are similar
c19 and sco are similar
c19 and coam are similar
c19 and mcp are similar
c19 and ibt are similar
c19 and nmr are similar
c19 and hfa are similar
c19 and 1:1 are similar
c19 and nk are similar
c19 and amp are similar
c19 and 3c are similar
c19 and tea are similar
c19 and upo ar

In [82]:
without_num

10618204                   zeaxanthin
10618209                  hemoprotein
10649237                      ethanol
10649449                       sterol
10653745                          pha
                      ...            
40572067                 amylosucrase
40572208                  nitrogenase
40573728    cucurbitanetypemogrosides
40577193                      ethanol
40579636                          pga
Name: Norm_product, Length: 15103, dtype: object

In [83]:
# Join normalized products to each article
normie = pd.concat([org_prod, without_num.groupby(level=0).agg(list)],axis = 1)

In [84]:
normie.head()

,Title,Abstract,Journal,Year,PMC_ID,DOI,Type,Author,Text,Product_Source,Doc_text,Product,Norm_product
10618204,Increased production of zeaxanthin and other p...,The psbAII locus was used as an integration pl...,Applied and environmental microbiology,2000,91786.0,10.1128/AEM.66.1.64-72.2000,"Journal Article,Research Support, U.S. Gov't, ...","[ForeName:D,LastName:Lagarde] [ForeName:L,Last...",None,title,production of zeaxanthin and,[zeaxanthin],[zeaxanthin]
10618209,Expression of Alcaligenes eutrophus flavohemop...,Expression of the vhb gene encoding hemoglobin...,Applied and environmental microbiology,2000,91791.0,10.1128/AEM.66.1.98-104.2000,"Journal Article,Research Support, Non-U.S. Gov't,","[ForeName:A D,LastName:Frey] [ForeName:J E,Las...",None,abstract,heme proteins production,[heme proteins],[hemoprotein]
10649237,Altered regulation of pyruvate kinase or co-ov...,Glycolytic fluxes in resting Escherichia coli ...,Biotechnology and bioengineering,2000,NaN,10.1002/(sici)1097-0290(20000305)67:5<623::aid...,"Journal Article,Research Support, Non-U.S. Gov't,","[ForeName:M,LastName:Emmerling] [ForeName:J E,...",None,abstract,ethanol production rates,[ethanol],[ethanol]
10649449,Cloning and characterization of the Yarrowia l...,The squalene synthase (SQS) gene encodes a key...,"Yeast (Chichester, England)",2000,NaN,10.1002/(SICI)1097-0061(200002)16:3<197::AID-Y...,"Journal Article,Research Support, Non-U.S. Gov...","[ForeName:S,LastName:Merkulov] [ForeName:F,Las...",None,abstract,sterol biosynthesis,[sterol],[sterol]
10653745,A novel genetically engineered pathway for syn...,A new pathway to synthesize poly(hydroxyalkano...,Applied and environmental microbiology,2000,91890.0,10.1128/AEM.66.2.739-743.2000,"Journal Article,Research Support, Non-U.S. Gov't,","[ForeName:S J,LastName:Liu] [ForeName:A,LastNa...",None,title,A genetically route for production of poly(hyd...,[poly(hydroxyalkanoic-acids)],[pha]


In [85]:
normie.shape

(14455, 13)

In [86]:
df = normie

In [87]:
import numpy as np
def clean_single_characters(item):
    """
    Remove single-character strings from lists, convert empty lists to NaN
    """
    if not isinstance(item, list):
        return item
    
    # Filter out single-character strings
    filtered_list = [x for x in item if isinstance(x, str) and len(x) > 1]
    
    # Return NaN if list becomes empty, otherwise return filtered list
    return np.nan if len(filtered_list) == 0 else filtered_list

# Apply the cleaning function to the column
df['Norm_product'] = df['Norm_product'].apply(clean_single_characters)

# Alternative: If you want to do it in-place without creating a new column
# df['Norm_product'] = df['Norm_product'].apply(clean_single_characters)

In [88]:
output_data = df 

In [89]:
df.Norm_product

10618204                   [zeaxanthin]
10618209                  [hemoprotein]
10649237                      [ethanol]
10649449                       [sterol]
10653745                          [pha]
                       ...             
40572067                 [amylosucrase]
40572208                  [nitrogenase]
40573728    [cucurbitanetypemogrosides]
40577193                      [ethanol]
40579636                          [pga]
Name: Norm_product, Length: 14455, dtype: object

In [90]:
(output_data.Norm_product.explode().dropna().str.len()==1).sum()

np.int64(0)

In [91]:
output_data.Norm_product.explode().dropna()[output_data.Norm_product.explode().dropna().str.len()==1]

Series([], Name: Norm_product, dtype: object)

In [92]:
output_data.dropna(subset=['Norm_product']).Product_Source.value_counts()

Product_Source
title        8266
abstract     4815
full_text    1317
Name: count, dtype: int64

In [93]:
check_save_file(df, output_file, 'Articles')

Saved file in: /Users/elisamarquez/Documents/PhD/2semestre/Engineering_db/Engineering_db/files/Output/Articles/filtered_metabolic_eng_articles_with_products_cleaned_norm_V_2025_09_30.json


In [94]:
df.Norm_product.explode().value_counts().head(59)

Norm_product
ethanol                 640
protein                 398
fuel                    334
butanol                 254
lactate                 218
succinate2              213
lipid                   161
fattyacid               144
butanediol              128
pha                     115
fattyacids              105
biomass                 105
carotenoid              103
polyketide               91
hydroxypropionate        88
lysine                   87
phb                      81
propanediol              80
carotene                 78
lycopene                 75
acetate                  75
astaxanthin              72
itaconate2               71
xylitol                  71
terpenoid                63
malate2                  62
riboflavin               62
aminoacids               61
antimicrobialagent       59
pyruvate                 57
glycerol                 56
enzyme                   56
diesel                   54
isoprenoid               54
enzymes                  54
ester  

In [95]:
df.Norm_product.explode().value_counts().index.str.contains('PHA').sum()

np.int64(0)

In [96]:
df.Norm_product.explode().value_counts().tail(59)

Norm_product
deoxyamphoteronolides              1
melibiose                          1
quercetinglucoside                 1
polyphosphatephosphate             1
astilbin                           1
kosinostatin                       1
sfgfpreporter                      1
ipc                                1
ip                                 1
acylesters                         1
polymersgammapga                   1
lavender                           1
scaledup                           1
enzymenitrogenase                  1
lyze                               1
cinnamylalcoholglucoside           1
pyridinealkaloids                  1
dses                               1
disinapoylglucose                  1
mberafter                          1
isomers                            1
chondroitinbiopolymers             1
dihydroartemisinicaldehyde         1
genomeantibiotics                  1
acetyltags                         1
blpalpha1beta2                     1
propylenediamine         

In [97]:
df.Product.explode().value_counts().loc[df.Product.explode().value_counts().index.str.contains('tca')]

Product
tca                           3
four-carbon tca               1
rtca                          1
tca clavulanate               1
tricarboxylate (tca)-cycle    1
Name: count, dtype: int64

In [98]:
df.Product.explode().value_counts().loc[df.Product.explode().value_counts().index.str.contains('polyketide')]

Product
polyketide                                 57
polyketides                                34
peptide-polyketide                          2
assembly-line polyketide                    1
ii polyketide                               1
peptides polyketides                        1
polyketide–terpenoid hybrids                1
full-length polyketides                     1
disorazol polyketides                       1
polyketide synthase/ peptide synthetase     1
polyketide starter unit                     1
ansamycin polyketide precursors             1
stilbene polyketides                        1
aromatic spiroketal polyketide              1
ansamycin polyketide precursor              1
polyketide 6-msa                            1
macrolide polyketides                       1
polyketide candidates                       1
aromatic polyketides                        1
polyketide-peptide                          1
antitumoral polyketide                      1
polyketide 6-deoxyerythron